### Import libraries

In [1]:
import os
import pickle
from methyldl.deconvolution.xgbdeconvolver import *
from tqdm import tqdm
from collections import defaultdict
import torch.nn as nn
from copy import deepcopy
from methyldl.deconvolution.deep_deconvolvers.training import train_matrix_deconvolver
import os.path as Path
from torch.utils.data import DataLoader, TensorDataset
from methyldl.deconvolution.least_squares_deconvolvers import (
    NNLSDeconvolver,
    PSLSDeconvolver,
)
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

### Configuring target model and extracting computed average scores

In [4]:
from edautils import *

reads_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsTrainingData_hg38_mincpg_4_minlen_10/"
dmr_label_column = "dmr_ctype_label"
mincpg_pointer = reads_data_path.find("mincpg_")
# classifier_model_path = f"../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_{dmr_label_column}_{reads_data_path[mincpg_pointer:]}"
classifier_model_path = "../Tutorials/archive/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels"
mincpg = int(reads_data_path[mincpg_pointer + 7 : mincpg_pointer + 8])
features_encoding = "extracted_numpy"
n_cell_types = num_dmr_groups = 39
n_pred_classes = 39 if "soft_labels" in classifier_model_path else 40

In [5]:
df = np.load(Path.join(classifier_model_path, "features_1.1_cutoff.npz"))

In [ ]:
features_test = df["features_test"]
features_train = df["features_train"]
features_valid = df["features_valid"]
target_proportions = df["proportions"]

### Defining deconvolvers

In [48]:
swn = nn.Sequential(
    nn.Linear(152, 1024),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(1024, 39),
    nn.Softmax(dim=-1),
)

mlp = nn.Sequential(
    nn.Linear(152, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(256, 152),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(152, 39),
    nn.Softmax(dim=-1),
)

xgb_config = XGBDeconvolverConfig(
    n_estimators=500,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=1,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=100,
    random_state=42,
)

xgb = XGBoostDeconvolver(
    config=xgb_config,
    output_transform="clip_normalize",
    n_dmr_groups=num_dmr_groups,
    n_pred_classes=n_pred_classes,
    n_cell_types=n_cell_types,
    with_reject_features=False,
    process_inputs=False,
)

nnls = NNLSDeconvolver()
psls = PSLSDeconvolver()

### Loading deconvolvers weights

In [49]:
device = "cuda"

In [50]:
swn.to(device=device)
swn.load_state_dict(
    torch.load(
        Path.join(classifier_model_path, "swn_mlp_best_deconvolver.pt"),
        weights_only=True,
    )
)
swn.eval()
mlp.to(device=device)
mlp.load_state_dict(
    torch.load(
        Path.join(classifier_model_path, "mlp_best_deconvolver.pt"), weights_only=True
    )
)
mlp.eval()
xgb = xgb.load(Path.join(classifier_model_path, "xgb_dpth15.joblib"))
nnls = nnls.load(Path.join(classifier_model_path, "nnls_deconvolver.joblib"))
psls = psls.load(Path.join(classifier_model_path, "psls_deconvolver.joblib"))

### Generating predictions

In [51]:
results = defaultdict(tuple)
model_tripples = [
    (swn, "swn", "nn"),
    (mlp, "mlp", "nn"),
    (xgb, "xgb", "xgb"),
    (nnls, "nnls", "ls"),
    (psls, "psls", "ls"),
]

In [59]:
def infer_multiple_deconvolvers(features, target_proportions, model_tripples):
    test_loader = DataLoader(
        TensorDataset(
            torch.FloatTensor(features), torch.FloatTensor(target_proportions)
        ),
        batch_size=2000,
    )
    results = defaultdict(tuple)
    for model, model_name, model_type in model_tripples:
        all_preds = []
        all_targets = target_proportions
        if model_type == "nn":
            all_targets = []
            with torch.no_grad():
                for X, y in test_loader:
                    X, y = X.to(device), y.to(device)
                    pred = model(X)
                    all_preds.append(pred.cpu())
                    all_targets.append(y.cpu())
            # Compute all metrics
            all_preds = torch.cat(all_preds, dim=0).numpy()
            all_targets = torch.cat(all_targets, dim=0)
            all_targets = all_targets.numpy()
        elif model_type == "xgb":
            all_preds = model._predict_raw(features)
            all_preds = model._transform_output(all_preds)
        elif model_type == "ls":
            if "nnls" in model_name:
                all_preds, _, _ = model.predict(features, n_workers=1)
            elif "psls" in model_name:
                all_preds = model.predict(features, n_workers=2)
        else:
            pass
        metrics = compute_deconvolution_metrics(all_preds, all_targets)
        all_preds = np.round(all_preds, 4)

        results[model_name] = (all_targets, all_preds, metrics)

    return results

In [ ]:
results_test = infer_multiple_deconvolvers(
    features=features_test,
    target_proportions=target_proportions,
    model_tripples=model_tripples,
)

In [55]:
import pandas as pd


def results_to_dataframe(results: dict, class_names: list = None) -> pd.DataFrame:
    """
    Convert the results dict into a DataFrame matching the supplementary table columns.

    Expected results structure:
        results[model_name] = (all_targets, all_preds, metrics_dict)

    model_name convention assumed (adjust parsing as needed):
        e.g. "hard_rej__dsimir__dirichlet__xgb__linear"
              labeling__classifier__clf_calib__deconvolver__dec_calib
    """
    rows = []
    for model_name, (targets, preds, metrics) in results.items():
        # --- parse model_name into table keys ---
        parts = model_name.split("__")
        if len(parts) == 5:
            labeling, classifier, clf_calib, deconvolver, dec_calib = parts
        elif len(parts) == 3:
            # UXM case: e.g. "uxm__uxm__linear"
            labeling, deconvolver, dec_calib = parts
            classifier = "---"
            clf_calib = "---"
        else:
            # fallback: store raw name, fill manually
            labeling = ""
            classifier = ""
            clf_calib = ""
            deconvolver = model_name
            dec_calib = ""

        worst_class = metrics["worst_class_name"]
        if class_names is not None and isinstance(worst_class, int):
            worst_class = class_names[worst_class]

        rows.append(
            {
                # "Labeling": labeling,
                # "Classifier": classifier,
                # "Clf. Calib.": clf_calib,
                "Deconvolver": deconvolver,
                # "Dec. Calib.": dec_calib,
                "R2": (
                    1.0 - metrics["mse"] / targets.var()
                    if hasattr(targets, "var")
                    else None
                ),
                "LoA": f"[{metrics['loa_lower']:.4f}, {metrics['loa_upper']:.4f}]",
                "LoA width": round(metrics["loa_width"], 4),
                "LoA (worst)": f"[{metrics['worst_class_loa_lower']:.4f}, {metrics['worst_class_loa_upper']:.4f}]",
                "LoA width (worst)": round(metrics["worst_class_loa_width"], 4),
                "Worst class": worst_class,
                "MSE": round(metrics["mse"], 6),
                "MAE": round(metrics["mae"], 6),
                "KL": round(metrics["kl"], 6),
                "Cosine Sim": round(metrics["cosine_sim"], 6),
            }
        )

    df = pd.DataFrame(rows)

    # Sort to match table grouping order
    df = df.sort_values(by=["Deconvolver"]).reset_index(drop=True)

    return df

In [ ]:
results_to_dataframe(results_test)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.976576,"[-0.0273, 0.0273]",0.0546,"[-0.0845, 0.0764]",0.1609,28,0.000194,0.004160,0.076462,0.986393
1,nnls,0.979957,"[-0.0253, 0.0253]",0.0505,"[-0.1163, 0.0895]",0.2058,28,0.000166,0.003300,0.080085,0.991102
2,psls,0.980120,"[-0.0251, 0.0251]",0.0503,"[-0.1172, 0.0898]",0.2070,28,0.000165,0.003185,0.076552,0.990876
3,swn,0.981938,"[-0.0240, 0.0240]",0.0479,"[-0.0949, 0.0791]",0.1740,28,0.000150,0.003248,0.054259,0.990031
4,xgb,0.969431,"[-0.0312, 0.0312]",0.0624,"[-0.1042, 0.0821]",0.1863,28,0.000253,0.003892,0.071319,0.989430


### Fitting linear callibrators

In [61]:
results_valid = infer_multiple_deconvolvers(
    features=features_valid,
    target_proportions=target_proportions,
    model_tripples=model_tripples,
)

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [02:24<00:00,  6.91it/s]


In [62]:
results_to_dataframe(results_valid)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.977715,"[-0.0266, 0.0266]",0.0533,"[-0.0713, 0.0675]",0.1388,28,0.000185,0.004265,0.079050,0.985948
1,nnls,0.977707,"[-0.0266, 0.0266]",0.0533,"[-0.1047, 0.0860]",0.1907,28,0.000185,0.003752,0.088899,0.989786
2,psls,0.979083,"[-0.0258, 0.0258]",0.0516,"[-0.1020, 0.0815]",0.1835,28,0.000173,0.003551,0.082573,0.990061
3,swn,0.978709,"[-0.0260, 0.0260]",0.0521,"[-0.0830, 0.0742]",0.1572,28,0.000176,0.003654,0.060873,0.988406
4,xgb,0.965313,"[-0.0332, 0.0332]",0.0664,"[-0.0707, 0.0877]",0.1584,29,0.000287,0.004253,0.077664,0.988058


In [70]:
def apply_callibration(results_valid, results_test, save_calibrators=False):
    results = defaultdict(tuple)
    for model_name in results_valid.keys():
        valid_preds = results_valid[model_name][0]
        val_target = results_valid[model_name][1]

        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.fit(valid_preds, val_target)
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

        if save_calibrators:
            calibrator.save_calibration_parameters(
                f"{classifier_model_path}/{model_name}_linear_calibrator.npz"
            )

    return results

In [73]:
calibrated_test_results = apply_callibration(
    results_valid, results_test, save_calibrators=True
)

In [74]:
results_to_dataframe(calibrated_test_results)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.978377,"[-0.0258, 0.0258]",0.0515,"[-0.0609, 0.0682]",0.1290,28,0.000173,0.004450,0.153173,0.987437
1,nnls,0.987026,"[-0.0194, 0.0194]",0.0388,"[-0.0495, 0.0628]",0.1122,28,0.000098,0.003849,0.133927,0.994771
2,psls,0.987163,"[-0.0194, 0.0194]",0.0389,"[-0.0527, 0.0649]",0.1175,28,0.000098,0.003642,0.129465,0.994890
3,swn,0.985723,"[-0.0210, 0.0210]",0.0420,"[-0.0568, 0.0674]",0.1242,28,0.000115,0.003476,0.106805,0.992416
4,xgb,0.978338,"[-0.0243, 0.0243]",0.0487,"[-0.0639, 0.0597]",0.1236,29,0.000154,0.004903,0.129503,0.992218
